<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m1_01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M1.01 — Who earns more than $10 an hour?

In the lab we explored the **adult census** dataset together. In this
workbook you repeat the same steps **on your own** with a new dataset, and
go a bit further.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The notebook stops at the first gap you have not
  filled, so "Run all" will not get you to the end. The error
  `NameError: name '____' is not defined` means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "age_constant": "e7f6c011776e8db7",
    "lazy_model_accuracy": "18db98e3760ea465",
    "most_categories": "3d7f19555f1d2f07",
    "n_features": "4a44dc15364204a8",
    "n_samples": "5ef6514ed3304cf6",
    "numerical_features": "8809d6cbbcfd05bc",
    "odd_row": "6d05621ab7cb7b4f",
    "share_high_wage": "a3b9afe536de2b8b",
    "target_column": "06aca62677f73f86"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Re-read the text above this cell and try again."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. The dataset

The data comes from the **1985 Current Population Survey** (CPS), a survey
of US workers. Each row is one person. The columns are:

| column | meaning |
|---|---|
| `EDUCATION` | number of years of education |
| `SOUTH` | does the person live in the South of the US? |
| `SEX` | sex of the person |
| `EXPERIENCE` | number of years of work experience |
| `UNION` | is the person a member of a trade union? |
| `WAGE` | **hourly wage, in dollars** |
| `AGE` | age in years |
| `RACE` | race of the person |
| `OCCUPATION` | type of job |
| `SECTOR` | sector of activity |
| `MARR` | marital status |

Our goal later in this module: **predict whether a person earns more than
$10 an hour** from the other columns. The column we want to predict is the
**target**.

✏️ Set `target_column` to the name of the column that holds the hourly wage
(as a string, e.g. `"AGE"`).

In [ ]:
import pandas as pd

wages = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/cps_85_wages.csv"
)
target_column = ____
check("target_column", target_column)
wages.head()

## 2. How big is it?

Run the next cell and read its output.

In [ ]:
wages.shape

✏️ `wages.shape` gives `(rows, columns)`. Fill in:

- `n_samples`: how many people are in the dataset;
- `n_features`: how many columns we can use **to predict** the wage. Careful:
  the target itself is not a feature.

In [ ]:
n_samples = ____
n_features = ____
check("n_samples", n_samples)
check("n_features", n_features)

## 3. Numerical or categorical?

A column is **numerical** when its values are numbers that can be compared
and averaged (for example 12 years of education is more than 8). It is
**categorical** when its values are labels from a finite set (for example
`Management`, `Sales`…).

In [ ]:
wages.dtypes

✏️ Using the output above, list the **numerical features** (not the target)
as a list of strings, e.g. `["A", "B"]`.

In [ ]:
numerical_features = ____
check("numerical_features", numerical_features)

For the categorical columns, it is useful to know how many different values
(**categories**) each one has:

In [ ]:
categorical_features = [
    c for c in wages.columns if c not in numerical_features + [target_column]
]
wages[categorical_features].nunique()

✏️ Which categorical column has the most categories? Write its name.

In [ ]:
most_categories = ____
check("most_categories", most_categories)

## 4. Building the target

We want a **classification** problem: "does this person earn more than $10
an hour?" (yes = `True`, no = `False`).

🔮 **Predict first.** Before running the computation, guess what share of
people earn more than $10 an hour. Choose one of `"less than 25%"`,
`"25-50%"`, `"50-75%"`, `"more than 75%"`.

In [ ]:
my_guess = ____
commit(my_guess, ["less than 25%", "25-50%", "50-75%", "more than 75%"])

✏️ Set `threshold` to the wage (in dollars per hour) that separates the two
classes, as described above. Then run the cell.

In [ ]:
threshold = ____
high_wage = wages[target_column] > threshold
high_wage.value_counts(normalize=True)

✏️ What share of people earn **more** than the threshold? Write it as a
number between 0 and 1, **rounded to 2 decimals** (e.g. `0.47`).

In [ ]:
share_high_wage = ____
check("share_high_wage", share_high_wage)

### ➕ Going further: a "model" that does not look at the data

Imagine a lazy model that ignores every feature and **always answers "no,
this person does not earn more than $10"**. It is right for every person
who indeed earns $10 or less.

✏️ What is the **accuracy** (share of correct answers) of this lazy model?
Round to 2 decimals.

In [ ]:
lazy_model_accuracy = ____
check("lazy_model_accuracy", lazy_model_accuracy)

🧠 **Think.** A classmate trains a model on this data and proudly reports
"70% accuracy!". Is that impressive? Keep your answer in mind: we come back
to this idea of a **baseline** in the next workbook.

## 5. Looking at the numerical features

Distributions of the numerical columns:

In [ ]:
import matplotlib.pyplot as plt

wages[numerical_features + [target_column]].hist(figsize=(10, 6), bins=20)
plt.tight_layout()

🧠 **Think.** Look at the `WAGE` histogram. Is it symmetric? Are there a few
people with a much higher wage than the others?

Now let us see how the numerical features relate to our target. Each dot is
a person, coloured by whether they earn more than $10:

In [ ]:
import seaborn as sns

sns.pairplot(
    wages[numerical_features].assign(high_wage=high_wage),
    hue="high_wage",
    plot_kws={"alpha": 0.4},
    height=2.5,
)

🧠 **Think.** On the diagonal, which **single** feature shows the clearest
difference between the two classes? Which one shows almost none?

### ➕ Going further: a suspicious relationship

In the pair plot, the panel `AGE` vs `EXPERIENCE` (and to some extent
`AGE` vs `EDUCATION`) looks very structured. Let us test an idea: maybe
experience is simply computed as `AGE - EDUCATION - (some constant)`.

In [ ]:
(wages["AGE"] - wages["EDUCATION"] - wages["EXPERIENCE"]).value_counts()

✏️ For almost every person, `AGE - EDUCATION - EXPERIENCE` is the same
number. Which one?

In [ ]:
age_constant = ____
check("age_constant", age_constant)

So for nearly everyone, `EXPERIENCE = AGE - EDUCATION - 6`: people are
assumed to start school at 6 and to work as soon as they leave school.
`EXPERIENCE` was **computed** by the survey, not measured.

✏️ One person does not follow this rule. Run the next cell and write the
**row index** (the number on the left) of that person.

In [ ]:
wages[wages["AGE"] - wages["EDUCATION"] - wages["EXPERIENCE"] != age_constant]

In [ ]:
odd_row = ____
check("odd_row", odd_row)

🧠 **Think.**

1. If `EXPERIENCE` can be computed exactly from `AGE` and `EDUCATION`, does
   it bring **new information** to a model?
2. The odd row: is it more likely a real person or a data entry error? What
   would you do with it?

## 6. Looking at the categorical features

Share of people earning more than $10 in each category:

In [ ]:
for column in categorical_features:
    print(high_wage.groupby(wages[column]).mean().round(2).sort_values(), "\n")

🧠 **Think.**

1. Which occupation has the highest share of high wages? The lowest?
2. The share of high wages differs between men and women. If a model uses
   `SEX` to predict the wage, what could go wrong if that model was used,
   for example, to propose salaries?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - what a 70%-accurate model is worth on this dataset, and why;
   - the `AGE` / `EDUCATION` / `EXPERIENCE` relationship and what it means
     for a model;
   - one thing in the categorical columns that surprised you.